# 📊 Exploratory Data Analysis — Insurance Enrollment Prediction

This notebook analyses the raw `employee_data.csv` to understand feature distributions,
correlations, class balance, and potential data quality issues before model training.

**Sections:**
1. Data Overview & Schema
2. Missing Values & Data Quality
3. Target Distribution (Class Balance)
4. Numerical Feature Analysis
5. Categorical Feature Analysis
6. Feature Correlations
7. Key Takeaways

In [ ]:
# --- Imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

# Load data
DATA_PATH = Path("../data/raw/employee_data.csv")
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Data Overview & Schema

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary of numerical features
df.describe()

In [ ]:
# Statistical summary of categorical features
df.describe(include="object")

## 2. Missing Values & Data Quality

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_summary[missing_summary["missing_count"] > 0])
if missing.sum() == 0:
    print("\n✅ No missing values found in any column.")

In [ ]:
# Check for duplicate rows
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

# Check for duplicate employee IDs
n_id_dupes = df["employee_id"].duplicated().sum()
print(f"Duplicate employee_ids: {n_id_dupes}")

## 3. Target Distribution (Class Balance)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_counts = df["enrolled"].value_counts()
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(target_counts.index.astype(str), target_counts.values, color=colors)
axes[0].set_xlabel("Enrolled")
axes[0].set_ylabel("Count")
axes[0].set_title("Target Class Distribution")
for i, (idx, val) in enumerate(zip(target_counts.index, target_counts.values)):
    axes[0].text(i, val + 50, str(val), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(
    target_counts.values,
    labels=["Not Enrolled (0)", "Enrolled (1)"],
    autopct="%1.1f%%",
    colors=colors,
    startangle=90,
    explode=(0.03, 0.03),
)
axes[1].set_title("Target Class Proportion")

plt.tight_layout()
plt.savefig("../data/processed/target_distribution.png", bbox_inches="tight")
plt.show()

print(f"\nClass ratio (enrolled/not): {target_counts[1]/target_counts[0]:.2f}")

## 4. Numerical Feature Analysis

In [ ]:
num_features = ["age", "salary", "tenure_years"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(num_features):
    sns.histplot(
        data=df, x=col, hue="enrolled", kde=True,
        ax=axes[i], palette=colors, alpha=0.6,
    )
    axes[i].set_title(f"{col} by Enrollment")

plt.tight_layout()
plt.savefig("../data/processed/numerical_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# Box plots to spot outliers
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(num_features):
    sns.boxplot(
        data=df, x="enrolled", y=col, ax=axes[i],
        palette=colors, width=0.4,
    )
    axes[i].set_title(f"{col} by Enrollment")
    axes[i].set_xticklabels(["Not Enrolled", "Enrolled"])

plt.tight_layout()
plt.savefig("../data/processed/numerical_boxplots.png", bbox_inches="tight")
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
cat_features = ["gender", "marital_status", "employment_type", "region", "has_dependents"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df["enrolled"], normalize="index") * 100
    ct.plot(
        kind="bar", stacked=True, ax=axes[i],
        color=colors, alpha=0.85,
    )
    axes[i].set_title(f"Enrollment Rate by {col}")
    axes[i].set_ylabel("Percentage")
    axes[i].legend(["Not Enrolled", "Enrolled"], loc="upper right")
    axes[i].tick_params(axis="x", rotation=0)

# Hide unused subplot
axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig("../data/processed/categorical_analysis.png", bbox_inches="tight")
plt.show()

## 6. Feature Correlations

In [ ]:
# Correlation heatmap for numerical features + target
corr_cols = num_features + ["enrolled"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".3f",
    cmap="coolwarm", center=0, linewidths=1,
    square=True, cbar_kws={"shrink": 0.8},
)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../data/processed/correlation_heatmap.png", bbox_inches="tight")
plt.show()

print("\nCorrelations with target (enrolled):")
print(corr_matrix["enrolled"].drop("enrolled").sort_values(ascending=False))

In [ ]:
# Pair plot for numerical features coloured by target
g = sns.pairplot(
    df[corr_cols], hue="enrolled",
    palette=colors, diag_kind="kde",
    plot_kws={"alpha": 0.4, "s": 15},
)
g.figure.suptitle("Pairwise Feature Relationships", y=1.02)
plt.tight_layout()
plt.savefig("../data/processed/pairplot.png", bbox_inches="tight")
plt.show()

## 7. Key Takeaways

Summarise observations here after running the notebook:

- **Class balance**: Check the enrollment split — if imbalanced, consider class weights or SMOTE.
- **Feature importance signals**: Note which features show clear separation by enrollment.
- **Outliers**: Review box plots for extreme salary or tenure values.
- **Correlations**: Low pairwise correlation among features is good (less multicollinearity).
- **Data quality**: Confirm no missing values or duplicates.